# Lesson 6: 音階と周波数

**コンパニオンノートブック** — 詳しい解説は本文 Lesson 6 を参照してください。

## セットアップ

In [ ]:
# --- 最初に1回だけ実行 ---
import sys
try:
    import google.colab
    !pip install -q japanize-matplotlib
    !git clone -q https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
except ImportError:
    sys.path.append('..')

from audio_lib.notebook import setup_environment
setup_environment()

## このレッスンで学ぶこと

- 平均律の仕組みを理解する
- MIDI 番号と周波数の変換式を学ぶ
- 音程と周波数比の関係を理解する
- Python でドレミファソラシドを生成して鳴らす


## 6.1 オクターブ — 周波数が2倍で「同じ音」

In [ ]:
import numpy as np
from IPython.display import display
from audio_lib import sine_wave
from audio_lib.notebook import play_sound

# A4（440Hz）と A5（880Hz）を聞き比べる
display(play_sound(sine_wave(440, 1.5), "A4: 440 Hz"))
display(play_sound(sine_wave(880, 1.5), "A5: 880 Hz（1オクターブ上）"))
display(play_sound(sine_wave(220, 1.5), "A3: 220 Hz（1オクターブ下）"))

### 「12等分」は等差でなく等比

In [ ]:
# 等差分割（間違い）と等比分割（正しい）の比較
f_low = 440   # A4
f_high = 880  # A5

# 等差分割: 440, 476.7, 513.3, ..., 880（等間隔に足す）
equal_add = np.linspace(f_low, f_high, 13)

# 等比分割: 440 × r^0, 440 × r^1, ..., 440 × r^12（等比率でかける）
r = 2 ** (1/12)
equal_mul = f_low * r ** np.arange(13)

print("等差分割（間違い）:")
print("  ", [f"{f:.1f}" for f in equal_add])
print()
print("等比分割（平均律）:")
print("  ", [f"{f:.1f}" for f in equal_mul])

### 半音比 $r = \sqrt[12]{2}$

In [ ]:
r = 2 ** (1/12)
print(f"半音の周波数比 r = 2^(1/12) = {r:.5f}")
print(f"検算: r^12 = {r**12:.6f}")

### 変換式

In [ ]:
from audio_lib import note_to_frequency

# いくつかの MIDI 番号で確認
for midi_num in [60, 69, 72, 81]:
    freq = note_to_frequency(midi_num)
    print(f"MIDI {midi_num:3d} → {freq:8.2f} Hz")

In [ ]:
from audio_lib import frequency_to_note

# 周波数から MIDI 番号を求める
for freq in [261.6, 440.0, 523.3, 880.0]:
    midi_num = frequency_to_note(freq)
    print(f"{freq:8.1f} Hz → MIDI {midi_num}")

### 変換式を自分で書いてみる

In [ ]:
def midi_to_freq(note_number):
    """MIDI ノート番号を周波数に変換"""
    return 440.0 * (2.0 ** ((note_number - 69) / 12.0))

def freq_to_midi(frequency):
    """周波数を MIDI ノート番号に変換（四捨五入）"""
    return round(12 * np.log2(frequency / 440.0) + 69)

# 検算
print(f"MIDI 69 → {midi_to_freq(69):.1f} Hz（A4 = 440 Hz）")
print(f"MIDI 60 → {midi_to_freq(60):.1f} Hz（C4）")
print(f"440 Hz → MIDI {freq_to_midi(440)}（A4）")
print(f"262 Hz → MIDI {freq_to_midi(262)}（C4）")

### MIDI 番号からスケールを作る

In [ ]:
from audio_lib import note_to_frequency, sine_wave
from audio_lib.notebook import play_sound

# C メジャースケールの MIDI 番号
c_major = [60, 62, 64, 65, 67, 69, 71, 72]
note_names = ["ド", "レ", "ミ", "ファ", "ソ", "ラ", "シ", "ド"]

dur = 0.5  # 各音の長さ（秒）

for midi_num, name in zip(c_major, note_names):
    freq = note_to_frequency(midi_num)
    sig = sine_wave(freq, dur)
    display(play_sound(sig, f"{name}（MIDI {midi_num}, {freq:.1f} Hz）"))

### スケールを1つの音声としてつなげる

In [ ]:
from audio_lib import AudioSignal

sample_rate = 44100
dur = 0.4

# 各音をサイン波で生成し、連結する
parts = []
for midi_num in c_major:
    freq = note_to_frequency(midi_num)
    sig = sine_wave(freq, dur)
    parts.append(sig.data)

# NumPy 配列を連結して1つの AudioSignal にする
scale_data = np.concatenate(parts)
scale_signal = AudioSignal(scale_data, sample_rate)

display(play_sound(scale_signal, "C メジャースケール"))

### 周波数の確認

In [ ]:
print(f"{'音名':>4} {'MIDI':>4} {'周波数':>10} {'前の音との比':>12}")
print("-" * 38)

prev_freq = None
for midi_num, name in zip(c_major, note_names):
    freq = note_to_frequency(midi_num)
    if prev_freq is not None:
        ratio = freq / prev_freq
        print(f"{name:>4} {midi_num:>4} {freq:>10.2f} Hz {ratio:>10.5f}")
    else:
        print(f"{name:>4} {midi_num:>4} {freq:>10.2f} Hz {'---':>10}")
    prev_freq = freq

### 主な音程の周波数比

In [ ]:
intervals = [
    ("ユニゾン", 0, "1:1"),
    ("短3度", 3, "6:5"),
    ("長3度", 4, "5:4"),
    ("完全4度", 5, "4:3"),
    ("完全5度", 7, "3:2"),
    ("オクターブ", 12, "2:1"),
]

print(f"{'音程':<10} {'半音数':>4} {'平均律の比':>10} {'整数比':>6}")
print("-" * 38)
for name, semitones, ratio_str in intervals:
    ratio = 2 ** (semitones / 12)
    print(f"{name:<10} {semitones:>4} {ratio:>10.5f} {ratio_str:>6}")

### 平均律と純正律のずれ

In [ ]:
f0 = 440  # A4

# 完全5度（A4 → E5）
f_equal = f0 * 2 ** (7/12)    # 平均律: 659.26 Hz
f_just  = f0 * 3 / 2          # 純正律: 660.00 Hz

print(f"完全5度: 平均律 {f_equal:.2f} Hz / 純正律 {f_just:.2f} Hz")
print(f"差: {abs(f_equal - f_just):.2f} Hz")

display(play_sound(sine_wave(f_equal, 2.0), f"平均律の完全5度: {f_equal:.1f} Hz"))
display(play_sound(sine_wave(f_just, 2.0),  f"純正律の完全5度: {f_just:.1f} Hz"))

In [ ]:
# 長3度（A4 → C#5）
f_equal_3 = f0 * 2 ** (4/12)   # 平均律: 554.37 Hz
f_just_3  = f0 * 5 / 4         # 純正律: 550.00 Hz

print(f"長3度: 平均律 {f_equal_3:.2f} Hz / 純正律 {f_just_3:.2f} Hz")
print(f"差: {abs(f_equal_3 - f_just_3):.2f} Hz")

display(play_sound(sine_wave(f_equal_3, 2.0), f"平均律の長3度: {f_equal_3:.1f} Hz"))
display(play_sound(sine_wave(f_just_3, 2.0),  f"純正律の長3度: {f_just_3:.1f} Hz"))

## 6.6 音名と MIDI 番号の対応

In [ ]:
from audio_lib import note_name_to_number, number_to_note_name

# 音名 → MIDI 番号
for name in ["C4", "A4", "G#3", "Bb5"]:
    num = note_name_to_number(name)
    print(f"{name:>4} → MIDI {num}")

print()

# MIDI 番号 → 音名
for num in [60, 69, 72, 48]:
    name = number_to_note_name(num)
    print(f"MIDI {num} → {name}")

### ピアノの全鍵盤と周波数

In [ ]:
import matplotlib.pyplot as plt

# ピアノの範囲: A0 (MIDI 21) 〜 C8 (MIDI 108)
midi_range = np.arange(21, 109)
freqs = [note_to_frequency(n) for n in midi_range]

plt.figure(figsize=(12, 5))
plt.plot(midi_range, freqs)
plt.xlabel("MIDI ノート番号")
plt.ylabel("周波数 (Hz)")
plt.title("MIDI ノート番号と周波数の関係")
plt.grid(True, alpha=0.3)

# オクターブごとに印をつける
for midi_num in [21, 33, 45, 57, 69, 81, 93, 105]:
    freq = note_to_frequency(midi_num)
    name = number_to_note_name(midi_num)
    plt.annotate(f"{name}\n{freq:.0f}Hz", (midi_num, freq),
                 textcoords="offset points", xytext=(0, 10),
                 ha='center', fontsize=8)

plt.tight_layout()
plt.show()

### 対数スケールで見ると

In [ ]:
plt.figure(figsize=(12, 5))
plt.semilogy(midi_range, freqs)
plt.xlabel("MIDI ノート番号")
plt.ylabel("周波数 (Hz)")
plt.title("MIDI ノート番号と周波数の関係（対数スケール）")
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 6.7 Lesson 5 の cutoff を振り返る

In [ ]:
print("Lesson 5 で使った cutoff の値:")
print(f"{'cutoff':>8} {'周波数':>10} {'音名':>6}")
print("-" * 30)

for cutoff in [60, 80, 100, 120, 130]:
    freq = note_to_frequency(cutoff)
    name = number_to_note_name(cutoff)
    print(f"{cutoff:>8} {freq:>10.1f} Hz {name:>6}")